# Phase 10.5C: Systematic Model Tuning & Experiment Registry

This notebook analyzes the results of systematic hyperparameter and architecture experiments recorded in the **Experiment Registry** (`data/models/experiments/`).

### Strict Protocol Constraints:
- **Zero Test Set Leakage**: All hyperparameter selection and strategy comparison evaluated exclusively on the training split using **3-Fold Expanding Chronological Cross-Validation**.
- **Feature Set**: Weather-Enriched v2 features (114 predictors).
- **Experiments Conducted**:
  1. **Ridge Alpha Grid** ($10^{-4}$ to $10^4$)
  2. **ElasticNet Regularization** ($L_1$ ratios $0.1, 0.5, 0.9$)
  3. **LightGBM Direct Multi-Output** (72 independent tree estimators)
  4. **LightGBM Horizon as Input Feature** ($X_t, h/72 \to y_{t+h}$)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.training_pipeline.experiment_registry import ExperimentRegistry

sns.set_theme(style="whitegrid")
registry = ExperimentRegistry()
df_leaderboard = registry.get_leaderboard()
df_leaderboard

## 1. Validation Leaderboard Analysis

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_leaderboard,
    x="Val RMSE (Mean)",
    y="Experiment ID",
    hue="Model",
    dodge=False,
    palette="viridis"
)
plt.title("Cross-Validation RMSE Comparison Across Systematic Experiments", fontsize=14, fontweight="bold")
plt.xlabel("Mean Validation RMSE (Lower is Better)")
plt.ylabel("Experiment ID")
plt.xlim(80, 105)
plt.tight_layout()
plt.show()

## 2. Ridge Regularization Sensitivity (Alpha Grid)

In [ ]:
ridge_exps = df_leaderboard[df_leaderboard["Model"] == "Ridge Regression"].copy()
# Extract alpha from notes
ridge_exps["alpha"] = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
ridge_exps = ridge_exps.sort_values("alpha")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(ridge_exps["alpha"], ridge_exps["Val RMSE (Mean)"], marker='o', color='b', linewidth=2, label='Mean Val RMSE')
ax1.set_xscale('log')
ax1.set_xlabel('Ridge Alpha (log scale)', fontsize=12)
ax1.set_ylabel('Mean Validation RMSE', color='b', fontsize=12)
ax1.tick_params(axis='y', labelcolor='b')
ax1.set_title('Ridge Performance vs. Regularization Strength Alpha', fontsize=14, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(ridge_exps["alpha"], ridge_exps["Val R²"], marker='s', color='g', linestyle='--', label='Val R²')
ax2.set_ylabel('Validation R²', color='g', fontsize=12)
ax2.tick_params(axis='y', labelcolor='g')

plt.tight_layout()
plt.show()

## 3. Horizon Breakdown: Ridge vs ElasticNet vs LightGBM

In [ ]:
selected_exps = ["EXP-005", "EXP-012", "EXP-013", "EXP-014"]
df_sel = df_leaderboard[df_leaderboard["Experiment ID"].isin(selected_exps)].copy()

horizon_data = []
for _, row in df_sel.iterrows():
    horizon_data.append({"Experiment": f"{row['Experiment ID']} ({row['Model']})", "Horizon": "h+1 (1h)", "RMSE": row["h+1 RMSE"]})
    horizon_data.append({"Experiment": f"{row['Experiment ID']} ({row['Model']})", "Horizon": "h+24 (1 Day)", "RMSE": row["h+24 RMSE"]})
    horizon_data.append({"Experiment": f"{row['Experiment ID']} ({row['Model']})", "Horizon": "h+72 (3 Days)", "RMSE": row["h+72 RMSE"]})

df_horizons = pd.DataFrame(horizon_data)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_horizons, x="Horizon", y="RMSE", hue="Experiment", palette="Set2")
plt.title("Per-Horizon RMSE (h+1, h+24, h+72) Across Top Candidates", fontsize=14, fontweight="bold")
plt.ylabel("Validation RMSE")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Key Scientific Conclusions from Phase 10.5C

1. **Ridge Regression Regularization Stability**: Performance is remarkably stable across $\alpha \in [10^{-4}, 100.0]$ (Mean Val RMSE ~ **85.74**, $R^2 = 0.5123$). Beyond $\alpha=1000$, heavy shrinkage causes bias ($RMSE=86.75$). $\alpha=1.0$ remains the optimal, highly robust regularizer.
2. **ElasticNet Comparison**: ElasticNet with high $L_1$ ratio ($l_1=0.9$) matches Ridge closely (**85.76 RMSE**), but is computationally heavier without providing metric advantages over $L_2$ Ridge.
3. **LightGBM Short vs Long Horizon Dynamics**:
   - **LightGBM Direct Multi-Output** achieves an extraordinary **46.18 RMSE** at $h+1$ (outperforming Ridge's $52.66$ by **12.3%** on immediate next-hour forecast).
   - However, at long horizons ($h+72$), error increases to $104.60$ (compared to Ridge's $92.39$), resulting in an overall mean RMSE of **90.86**.
4. **Training-Only Winner Locked**: **Ridge Regression (\(\alpha=1.0\)) with Weather Features** selected as the primary champion for overall multi-horizon stability, with LightGBM identified as an ultra-strong short-horizon specialist.